# OXXO V7.2



In [6]:

import os, re, json, time, unicodedata
from pathlib import Path
from functools import lru_cache

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

# CONFIG
TARGET_PLANOGRUPO = "Refrescos"
TARGET_DIRECTION = "ID"
TARGET_DESIGN_REF = "BCO_CF_Refrescos_3.5ID"   # canonicalized exact design
TARGET_SEGMENT = "BCO"

USE_CHAROLA_MODEL = False
OUTPUT_DIR = "oxxo_v7_2_outputs"

# For large instances, exact only where it makes sense
MAX_EXACT_BLOCK = 9
WEIGHTS = {
    "adjacency": 2.0,
    "precedence": 1.0,
    "anchor_gap": 0.2,
}

# UTILS
def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

def norm_text(s):
    if pd.isna(s):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    return s.strip()

def norm_col(c):
    c = norm_text(c).upper()
    c = c.replace("/", "_").replace("-", "_")
    c = re.sub(r"[^A-Z0-9_]+", "_", c)
    c = re.sub(r"_+", "_", c).strip("_")
    return c

def canonical_col(df, candidates, new_name):
    for c in candidates:
        if c in df.columns:
            return df.rename(columns={c: new_name})
    return df

def choose_existing(path_candidates):
    for p in path_candidates:
        if Path(p).exists():
            return Path(p)
    raise FileNotFoundError(f"No encontré ninguno de estos archivos: {path_candidates}")

def read_csv_robust(path, **kwargs):
    encodings = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False, **kwargs)
        except Exception as e:
            last_err = e
    raise last_err

# LOAD
def load_sources(base_dir="."):
    base = Path(base_dir)
    ejemplo = read_csv_robust(choose_existing([base/"Ejemplo.csv", base/"ejemplo.csv"]))
    oxxo1 = read_csv_robust(choose_existing([base/"oxxo_1.csv"]))
    plano = read_csv_robust(choose_existing([base/"ejemplo_planograma.csv"]))
    caso = pd.read_excel(choose_existing([base/"Caso de estudio TEC.xlsx"]))
    return ejemplo, oxxo1, plano, caso


# FEATURE EXTRACTION
def infer_provider_brand(desc):
    s = norm_text(desc).upper()
    provider, brand = "OTROS", "OTROS"

    rules = [
        (["COCA COLA", "COCACOLA"], ("COCA_COLA_FEMSA", "COCA_COLA")),
        (["SPRITE"], ("COCA_COLA_FEMSA", "SPRITE")),
        (["FANTA"], ("COCA_COLA_FEMSA", "FANTA")),
        (["MUNDET"], ("COCA_COLA_FEMSA", "MUNDET")),
        (["FRESCA"], ("COCA_COLA_FEMSA", "FRESCA")),
        (["DEL VALLE"], ("COCA_COLA_FEMSA", "DEL_VALLE")),
        (["POWERADE"], ("COCA_COLA_FEMSA", "POWERADE")),
        (["PEPSI"], ("PEPSICO", "PEPSI")),
        (["7UP"], ("PEPSICO", "7UP")),
        (["MIRINDA"], ("PEPSICO", "MIRINDA")),
        (["MANZANITA SOL"], ("PEPSICO", "MANZANITA_SOL")),
        (["SQUIRT"], ("PEPSICO", "SQUIRT")),
        (["EPURA"], ("PEPSICO", "EPURA")),
        (["PEAFIEL", "PENAFIEL"], ("PENAFIEL", "PENAFIEL")),
        (["CLAMATO"], ("PENAFIEL", "CLAMATO")),
        (["JARRITOS"], ("NOVAMEX", "JARRITOS")),
        (["SANGRIA SENORIAL"], ("NOVAMEX", "SANGRIA_SENORIAL")),
    ]
    for pats, vals in rules:
        if any(p in s for p in pats):
            provider, brand = vals
            break

    pkg = "OTRO"
    if "LATA" in s or "CAN" in s:
        pkg = "LATA"
    elif "VIDRIO" in s or "RETORN" in s:
        pkg = "VIDRIO_RET"
    elif "PET" in s or "PLAST" in s:
        pkg = "PET"

    is_zero = 1 if ("ZERO" in s or "LIGHT" in s or "SIN AZUCAR" in s) else 0
    size_ml = np.nan
    m = re.search(r"(\d+(?:\.\d+)?)\s*(ML|L)", s)
    if m:
        v = float(m.group(1))
        size_ml = v * 1000 if m.group(2) == "L" else v

    return provider, brand, pkg, is_zero, size_ml

# STANDARDIZE
def standardize_hist(df):
    df = df.copy()
    df.columns = [norm_col(c) for c in df.columns]

    df = canonical_col(df, ["PLANOGRUPO_DESC", "PLANOGRUPO"], "PLANOGRUPO")
    df = canonical_col(df, ["TAMANO_DESC", "TAMANO", "TAMANO_POST"], "TAMANO")
    df = canonical_col(df, ["ITEM_DESC", "ITEM"], "ITEM_DESC")
    df = canonical_col(df, ["UPC_CVE", "UPC", "SKU"], "UPC_CVE")
    df = canonical_col(df, ["NUM_FRENTES", "FRENTES"], "NUM_FRENTES")
    df = canonical_col(df, ["UBICACION_BANDEJA", "BANDEJA", "POSICION"], "UBICACION_BANDEJA")
    df = canonical_col(df, ["DIRECCION_LEGO_ID", "DIRECCION"], "DIRECCION_LEGO_ID")
    df = canonical_col(df, ["ANCHO", "WIDTH_PROD"], "ANCHO")
    df = canonical_col(df, ["ALTO", "HEIGHT_PROD"], "ALTO")
    df = canonical_col(df, ["PROFUNDO", "DEPTH_PROD"], "PROFUNDO")
    df = canonical_col(df, ["MUEBLE_ID", "MUEBLE"], "MUEBLE_ID")
    df = canonical_col(df, ["SEGMENTO_ID", "SEGMENTO"], "SEGMENTO_ID")

    required_text = ["SEGMENTO_ID","MUEBLE_ID","PLANOGRUPO","TAMANO","DIRECCION_LEGO_ID","UPC_CVE","ITEM_DESC"]
    required_num = ["NUM_FRENTES","CHAROLA","UBICACION_BANDEJA","ANCHO","ALTO","PROFUNDO"]
    for c in required_text:
        if c not in df.columns:
            df[c] = ""
    for c in required_num:
        if c not in df.columns:
            df[c] = np.nan

    for c in required_text:
        df[c] = df[c].map(norm_text)
    for c in required_num:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["DIRECCION_LEGO_ID"] = df["DIRECCION_LEGO_ID"].replace({"I_D":"ID","D_I":"DI"})
    aux = df["ITEM_DESC"].fillna(df["UPC_CVE"]).apply(infer_provider_brand)
    df[["provider","brand","pkg","is_zero","size_ml"]] = pd.DataFrame(aux.tolist(), index=df.index)

    df["product_key"] = df["UPC_CVE"].astype(str)
    df["format_key"] = df["SEGMENTO_ID"] + "|" + df["MUEBLE_ID"] + "|" + df["PLANOGRUPO"] + "|" + df["TAMANO"] + "|" + df["DIRECCION_LEGO_ID"]
    df["design_ref"] = df["SEGMENTO_ID"] + "_" + df["MUEBLE_ID"] + "_" + df["PLANOGRUPO"] + "_" + df["TAMANO"] + df["DIRECCION_LEGO_ID"]
    return df

def standardize_geometry(plano, caso):
    def _std(df):
        out = df.copy()
        out.columns = [norm_col(c) for c in out.columns]
        out = canonical_col(out, ["TAMANO_POST", "TAMANO"], "TAMANO")
        out = canonical_col(out, ["DIRECCION_LEGO_ID", "DIRECCION"], "DIRECCION_LEGO_ID")
        out = canonical_col(out, ["MUEBLE_ID", "MUEBLE"], "MUEBLE_ID")
        out = canonical_col(out, ["PLANOGRUPO_DESC", "PLANOGRUPO"], "PLANOGRUPO")
        for c in ["MUEBLE_ID","PLANOGRUPO","TAMANO","DIRECCION_LEGO_ID"]:
            if c not in out.columns:
                out[c] = ""
            out[c] = out[c].map(norm_text)
        for c in ["CHAROLA","WIDTH","HEIGHT","X","Y"]:
            if c not in out.columns:
                out[c] = np.nan
            out[c] = pd.to_numeric(out[c], errors="coerce")
        return out

    plano = _std(plano); caso = _std(caso)
    caso["source"] = "caso"; plano["source"] = "plano"
    geom = pd.concat([caso, plano], ignore_index=True, sort=False)
    geom = geom.sort_values(["source"]).drop_duplicates(
        subset=["MUEBLE_ID","PLANOGRUPO","TAMANO","DIRECCION_LEGO_ID","CHAROLA"],
        keep="first"
    )
    return geom

# TARGET INSTANCE
def build_target_instance(hist, geom):
    target = hist[hist["design_ref"] == TARGET_DESIGN_REF].copy()

    if target.empty:
        # fallback by components
        target = hist[
            (hist["SEGMENTO_ID"] == TARGET_SEGMENT) &
            (hist["PLANOGRUPO"] == TARGET_PLANOGRUPO) &
            (hist["DIRECCION_LEGO_ID"] == TARGET_DIRECTION) &
            (hist["MUEBLE_ID"] == "CF") &
            (hist["TAMANO"] == "3.5")
        ].copy()
        if target.empty:
            raise ValueError(f"No encontré target exacto ni fallback para {TARGET_DESIGN_REF}")

    grp = (
        target.groupby(["product_key"], as_index=False)
              .agg(
                  provider=("provider", lambda s: s.mode().iloc[0] if s.notna().any() else "OTROS"),
                  brand=("brand", lambda s: s.mode().iloc[0] if s.notna().any() else "OTROS"),
                  pkg=("pkg", lambda s: s.mode().iloc[0] if s.notna().any() else "OTRO"),
                  is_zero=("is_zero", "max"),
                  size_ml=("size_ml", "mean"),
                  ITEM_DESC=("ITEM_DESC", "first"),
                  NUM_FRENTES=("NUM_FRENTES", "mean"),
                  ANCHO=("ANCHO", "mean"),
                  ALTO=("ALTO", "mean"),
                  actual_charola=("CHAROLA", lambda s: s.mode().iloc[0] if s.notna().any() else np.nan),
                  actual_slot=("UBICACION_BANDEJA", lambda s: s.mode().iloc[0] if s.notna().any() else np.nan),
              )
    )
    grp["row_id"] = np.arange(1, len(grp)+1)

    first = target.iloc[0]
    g = geom[
        (geom["PLANOGRUPO"] == TARGET_PLANOGRUPO) &
        (geom["MUEBLE_ID"] == first["MUEBLE_ID"]) &
        (geom["TAMANO"] == first["TAMANO"]) &
        (geom["DIRECCION_LEGO_ID"] == TARGET_DIRECTION)
    ].copy()

    g = g[["CHAROLA","WIDTH","HEIGHT","X","Y"]].drop_duplicates().sort_values("CHAROLA")
    return grp, g, target

# TABLES
def build_tables(hist, target_hist):
    # full-history agg for optional charola model
    agg = (
        hist.groupby(["format_key","SEGMENTO_ID","MUEBLE_ID","PLANOGRUPO","TAMANO","DIRECCION_LEGO_ID",
                      "product_key","provider","brand","pkg","is_zero","size_ml"], dropna=False)
            .agg(
                char_modal=("CHAROLA", lambda s: s.mode().iloc[0] if s.notna().any() else np.nan),
                fronts_avg=("NUM_FRENTES","mean"),
                width_avg=("ANCHO","mean"),
            )
            .reset_index()
    )

    # target-design tables (precision-first)
    pc = target_hist.groupby(["product_key","CHAROLA"]).size().rename("cnt").reset_index()
    pc["prob"] = pc["cnt"] / pc.groupby("product_key")["cnt"].transform("sum")

    anchor_item = target_hist.groupby(["product_key","CHAROLA"])["UBICACION_BANDEJA"].mean().rename("anchor_item").reset_index()
    anchor_brand = target_hist.groupby(["brand","CHAROLA"])["UBICACION_BANDEJA"].mean().rename("anchor_brand").reset_index()
    anchor_provider = target_hist.groupby(["provider","CHAROLA"])["UBICACION_BANDEJA"].mean().rename("anchor_provider").reset_index()

    prec_rows, adj_rows = [], []
    for ch, g in target_hist.dropna(subset=["CHAROLA","UBICACION_BANDEJA"]).groupby("CHAROLA"):
        g = g.sort_values("UBICACION_BANDEJA")
        items = list(g["product_key"])
        for i in range(len(items)):
            for j in range(i+1, len(items)):
                prec_rows.append((items[i], items[j], int(ch), 1))
        for i in range(len(items)-1):
            a, b = items[i], items[i+1]
            adj_rows.append((a, b, int(ch), 1))
            adj_rows.append((b, a, int(ch), 1))

    precedence = pd.DataFrame(prec_rows, columns=["i","j","CHAROLA","cnt"])
    adjacency = pd.DataFrame(adj_rows, columns=["i","j","CHAROLA","cnt"])
    if len(precedence):
        precedence = precedence.groupby(["i","j","CHAROLA"], as_index=False)["cnt"].sum()
    if len(adjacency):
        adjacency = adjacency.groupby(["i","j","CHAROLA"], as_index=False)["cnt"].sum()

    return {
        "agg": agg,
        "pc": pc,
        "anchor_item": anchor_item,
        "anchor_brand": anchor_brand,
        "anchor_provider": anchor_provider,
        "precedence": precedence,
        "adjacency": adjacency,
    }

# CHAROLA MODEL
def train_charola_model(agg):
    df = agg.dropna(subset=["char_modal"]).copy()
    df["y"] = df["char_modal"].astype(int)

    cat_cols = ["SEGMENTO_ID","MUEBLE_ID","PLANOGRUPO","TAMANO","DIRECCION_LEGO_ID","provider","brand","pkg","product_key"]
    num_cols = ["fronts_avg","width_avg","size_ml","is_zero"]

    for c in cat_cols:
        if c not in df.columns:
            df[c] = ""
        df[c] = df[c].astype(str)
    for c in num_cols:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")

    pre = ColumnTransformer([
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols),
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median"))
        ]), num_cols)
    ])

    model = HistGradientBoostingClassifier(
        max_depth=5,
        learning_rate=0.08,
        max_iter=150,
        early_stopping=False
    )

    X = df[cat_cols + num_cols]
    Xt = pre.fit_transform(X)
    if hasattr(Xt, "toarray"):
        Xt = Xt.toarray()
    model.fit(Xt, df["y"])
    return {"pre": pre, "model": model, "cat_cols": cat_cols, "num_cols": num_cols}

def score_charolas(target_products, target_geom, tables, model_bundle=None):
    chars = target_geom["CHAROLA"].dropna().astype(int).tolist()
    pc_map = {(r.product_key, int(r.CHAROLA)): float(r.prob) for r in tables["pc"].itertuples()}

    model_probs = {}
    if USE_CHAROLA_MODEL and model_bundle is not None:
        df = target_products.copy()
        for c in model_bundle["cat_cols"]:
            if c not in df.columns:
                df[c] = ""
            df[c] = df[c].astype(str)
        for c in model_bundle["num_cols"]:
            if c not in df.columns:
                df[c] = np.nan
        Xt = model_bundle["pre"].transform(df[model_bundle["cat_cols"] + model_bundle["num_cols"]])
        if hasattr(Xt, "toarray"):
            Xt = Xt.toarray()
        proba = model_bundle["model"].predict_proba(Xt)
        classes = model_bundle["model"].classes_
        for i, rid in enumerate(df["row_id"].tolist()):
            model_probs[rid] = {int(c): float(p) for c, p in zip(classes, proba[i])}

    rows = []
    for r in target_products.itertuples():
        scores = {}
        for ch in chars:
            s_hist = pc_map.get((r.product_key, ch), 0.0)
            s_model = model_probs.get(r.row_id, {}).get(ch, 0.0)
            s = 0.8 * s_hist + 0.2 * s_model if USE_CHAROLA_MODEL else s_hist
            scores[ch] = s
        rows.append({"row_id": r.row_id, "charola_scores": scores})
    return pd.DataFrame(rows)

def assign_charolas(target_products, scores_df):
    out = target_products.merge(scores_df, on="row_id", how="left").copy()
    out["pred_charola"] = out["charola_scores"].map(lambda d: max(d, key=d.get) if isinstance(d, dict) and d else np.nan)
    return out

# ORDER / SLOT
def build_maps_for_charola(ch, tables):
    ai = {(r.product_key): float(r.anchor_item) for r in tables["anchor_item"][tables["anchor_item"]["CHAROLA"] == ch].itertuples()}
    ab = {(r.brand): float(r.anchor_brand) for r in tables["anchor_brand"][tables["anchor_brand"]["CHAROLA"] == ch].itertuples()}
    ap = {(r.provider): float(r.anchor_provider) for r in tables["anchor_provider"][tables["anchor_provider"]["CHAROLA"] == ch].itertuples()}
    prec = {(r.i, r.j): float(r.cnt) for r in tables["precedence"][tables["precedence"]["CHAROLA"] == ch].itertuples()}
    adj = {(r.i, r.j): float(r.cnt) for r in tables["adjacency"][tables["adjacency"]["CHAROLA"] == ch].itertuples()}
    return ai, ab, ap, prec, adj

def solve_small_block_exact(df_block, ch, ai_map, prec_map, adj_map):
    df = df_block.copy().reset_index(drop=True)
    items = df["product_key"].tolist()
    anchors = [float(ai_map.get(k, 999.0)) for k in items]
    n = len(df)

    @lru_cache(None)
    def dp(mask, last):
        if mask == (1 << n) - 1:
            return (0.0, ())
        best_score = None
        best_perm = None
        for j in range(n):
            if (mask >> j) & 1:
                continue
            score = 0.0
            if last != -1:
                score += WEIGHTS["adjacency"] * adj_map.get((items[last], items[j]), 0.0)
                score += WEIGHTS["precedence"] * prec_map.get((items[last], items[j]), 0.0)
                score -= WEIGHTS["anchor_gap"] * abs(anchors[last] - anchors[j])
            child_score, child_perm = dp(mask | (1 << j), j)
            total = score + child_score
            if best_score is None or total > best_score:
                best_score = total
                best_perm = (j,) + child_perm
        return best_score, best_perm

    _, perm = dp(0, -1)
    return df.iloc[list(perm)].copy().reset_index(drop=True)

def improve_large_block_local(df_block, ch, ai_map, prec_map, adj_map, max_passes=3):
    out = df_block.copy()
    out["anchor_item"] = out["product_key"].map(lambda k: ai_map.get(k, 999.0))
    out = out.sort_values(["anchor_item","NUM_FRENTES","ITEM_DESC"]).reset_index(drop=True)

    def seq_score(df):
        score = 0.0
        items = df["product_key"].tolist()
        anchors = df["anchor_item"].tolist()
        for i in range(len(items)-1):
            score += WEIGHTS["adjacency"] * adj_map.get((items[i], items[i+1]), 0.0)
            score += WEIGHTS["precedence"] * prec_map.get((items[i], items[i+1]), 0.0)
            score -= WEIGHTS["anchor_gap"] * abs(anchors[i] - anchors[i+1])
        return score

    cur_score = seq_score(out)
    for _ in range(max_passes):
        improved = False
        for i in range(len(out)-1):
            cand = out.copy()
            a = cand.iloc[i].copy()
            b = cand.iloc[i+1].copy()
            cand.iloc[i] = b
            cand.iloc[i+1] = a
            s = seq_score(cand)
            if s > cur_score:
                out = cand.reset_index(drop=True)
                cur_score = s
                improved = True
        if not improved:
            break
    return out.drop(columns=["anchor_item"], errors="ignore")

def assign_slots_absolute(df_char, ch, ai_map):
    out = df_char.copy().reset_index(drop=True)
    out["pred_ubicacion_bandeja"] = out["product_key"].map(lambda k: int(round(ai_map.get(k, 1.0))))
    return out

def solve_charola_hybrid(df_char, ch, tables):
    ai_map, ab_map, ap_map, prec_map, adj_map = build_maps_for_charola(ch, tables)
    g = df_char.copy()

    # provider order
    provider_order = sorted(g["provider"].fillna("OTROS").unique().tolist(), key=lambda p: ap_map.get(p, 999.0))
    pieces = []
    for prov in provider_order:
        gp = g[g["provider"] == prov].copy()
        brand_order = sorted(gp["brand"].fillna("OTROS").unique().tolist(), key=lambda b: ab_map.get(b, 999.0))
        for br in brand_order:
            gb = gp[gp["brand"] == br].copy()
            if len(gb) <= MAX_EXACT_BLOCK:
                pieces.append(solve_small_block_exact(gb, ch, ai_map, prec_map, adj_map))
            else:
                pieces.append(improve_large_block_local(gb, ch, ai_map, prec_map, adj_map))
    if len(pieces):
        ordered = pd.concat(pieces, ignore_index=True)
    else:
        ordered = g.copy().reset_index(drop=True)

    # absolute-slot prediction, not ordinal
    ordered = assign_slots_absolute(ordered, ch, ai_map)
    return ordered

# EVAL
def provider_blocks_metric(pred_df):
    rows = []
    for ch, g in pred_df.sort_values(["pred_charola","pred_ubicacion_bandeja","provider","brand"]).groupby("pred_charola"):
        prov = g["provider"].astype(str).tolist()
        if not prov:
            continue
        blocks = 1
        for i in range(1, len(prov)):
            if prov[i] != prov[i-1]:
                blocks += 1
        rows.append({"pred_charola": int(ch), "n_bloques_proveedor": blocks, "n_productos": len(g)})
    return pd.DataFrame(rows)

def evaluate_target(pred_df, target_df):
    df = pred_df.copy()
    need = ["row_id","actual_charola","actual_slot","provider","brand"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        df = df.merge(target_df[need], on="row_id", how="left")

    df["char_ok"] = (df["pred_charola"].astype(int) == df["actual_charola"].astype(int)).astype(int)
    df["slot_ok"] = (df["pred_ubicacion_bandeja"].astype(int) == df["actual_slot"].astype(int)).astype(int)
    df["slot_close"] = (abs(df["pred_ubicacion_bandeja"].astype(int) - df["actual_slot"].astype(int)) <= 1).astype(int)
    df["slot_abs_error"] = abs(df["pred_ubicacion_bandeja"].astype(int) - df["actual_slot"].astype(int))

    met = {
        "n_productos_predichos": int(len(df)),
        "charola_accuracy": float(df["char_ok"].mean()),
        "slot_accuracy": float(df["slot_ok"].mean()),
        "slot_close_accuracy": float(df["slot_close"].mean()),
        "slot_abs_error_mean": float(df["slot_abs_error"].mean()),
        "slot_abs_error_median": float(df["slot_abs_error"].median()),
    }
    cond = df[df["char_ok"] == 1]
    if len(cond):
        met["slot_accuracy_given_charola_ok"] = float(cond["slot_ok"].mean())
        met["slot_close_given_charola_ok"] = float(cond["slot_close"].mean())
        met["slot_abs_error_mean_given_charola_ok"] = float(cond["slot_abs_error"].mean())
    else:
        met["slot_accuracy_given_charola_ok"] = np.nan
        met["slot_close_given_charola_ok"] = np.nan
        met["slot_abs_error_mean_given_charola_ok"] = np.nan

    pb = provider_blocks_metric(df)
    met["promedio_bloques_proveedor_por_charola"] = float(pb["n_bloques_proveedor"].mean()) if len(pb) else np.nan
    return met, df, pb

# VISUALIZACIÓN DEL PLANOGRAMA
def visualize_planogram(pred_df, target_geom, outdir):
    """
    Genera imagen PNG del planograma recomendado.
    Cada charola se dibuja como una fila de productos (izquierda→derecha por slot).
    Ancho de cada producto ∝ NUM_FRENTES × ANCHO físico.
    Colores por proveedor.
    """
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        import matplotlib.patches as mpatches
        from matplotlib.patches import FancyBboxPatch
    except ImportError:
        log("   [SKIP] matplotlib no disponible, instálalo con: pip install matplotlib")
        return None

    # ── Paleta de proveedores ─────────────────────────────────────────────
    PROVIDER_COLORS = {
        "COCA_COLA_FEMSA": "#CC0000",
        "PEPSICO":         "#003087",
        "PENAFIEL":        "#2E7D32",
        "NOVAMEX":         "#E65100",
        "OTROS":           "#546E7A",
    }
    def get_color(prov):
        return PROVIDER_COLORS.get(str(prov), "#546E7A")

    # ── Preparar datos ────────────────────────────────────────────────────
    df = pred_df.copy()
    df = df.dropna(subset=["pred_charola", "pred_ubicacion_bandeja"])
    df["pred_charola"]           = df["pred_charola"].astype(int)
    df["pred_ubicacion_bandeja"] = df["pred_ubicacion_bandeja"].astype(int)
    df["NUM_FRENTES"]            = pd.to_numeric(df.get("NUM_FRENTES"), errors="coerce").fillna(1).clip(lower=1)
    df["ANCHO"]                  = pd.to_numeric(df.get("ANCHO"),       errors="coerce").fillna(8)
    df["size_ml"]                = pd.to_numeric(df.get("size_ml"),     errors="coerce")

    charolas = sorted(df["pred_charola"].unique())
    n_ch = len(charolas)
    if n_ch == 0:
        log("   [SKIP] Sin charolas para dibujar.")
        return None

    # ── Layout del canvas ─────────────────────────────────────────────────
    ROW_H    = 2.8        # pulgadas por charola
    FIG_W    = 22
    FIG_H    = n_ch * ROW_H + 1.8
    CANVAS_W = 100.0      # unidades internas de ancho

    fig, axes = plt.subplots(
        n_ch, 1,
        figsize=(FIG_W, FIG_H),
        gridspec_kw={"hspace": 0.06}
    )
    if n_ch == 1:
        axes = [axes]

    fig.patch.set_facecolor("#EBEBEB")
    fig.suptitle(
        f"Planograma Recomendado  ·  {TARGET_DESIGN_REF}",
        fontsize=13, fontweight="bold", y=1.005, color="#1A1A1A"
    )

    # ── Dibujar charolas (arriba = charola más alta en el mueble) ─────────
    for ax_idx, ch in enumerate(reversed(charolas)):
        ax  = axes[ax_idx]
        sub = df[df["pred_charola"] == ch].sort_values("pred_ubicacion_bandeja")

        # Escalar anchos efectivos al canvas
        anchos_raw = (sub["ANCHO"] * sub["NUM_FRENTES"]).values
        total_w    = anchos_raw.sum()
        scale      = (CANVAS_W * 0.96) / max(total_w, 1)
        anchos_sc  = anchos_raw * scale

        # Fondo de la charola — alternar gris claro/blanco
        ax.set_facecolor("#DEDEDE" if ax_idx % 2 == 0 else "#E8E8E8")
        ax.set_xlim(0, CANVAS_W)
        ax.set_ylim(0, 1)
        ax.set_yticks([])
        ax.set_xticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)

        # Etiqueta lateral de charola
        ax.text(-1.8, 0.5, f"Charola\n{ch}",
                ha="right", va="center",
                fontsize=9, fontweight="bold", color="#333333")

        # Tablón de estante (línea inferior marrón)
        ax.axhline(0.045, color="#8B6914", linewidth=4, zorder=1)

        x = 2.0
        prev_prov = None

        for i, (_, row) in enumerate(sub.iterrows()):
            w       = max(float(anchos_sc[i]), 1.2)
            frentes = int(row.get("NUM_FRENTES", 1))
            prov    = str(row.get("provider", "OTROS"))
            brand   = str(row.get("brand", "")).replace("_", " ")
            size    = row.get("size_ml", np.nan)
            pkg     = str(row.get("pkg", ""))
            color   = get_color(prov)

            # Línea divisora entre proveedores distintos
            if prev_prov is not None and prov != prev_prov:
                ax.plot([x, x], [0.07, 0.93],
                        color="#FFFFFF", linewidth=2.0, zorder=5)
            prev_prov = prov

            # Rectángulo del producto
            rect = FancyBboxPatch(
                (x, 0.08), w - 0.25, 0.84,
                boxstyle="round,pad=0.02",
                facecolor=color, edgecolor="white",
                linewidth=0.8, alpha=0.92, zorder=2
            )
            ax.add_patch(rect)

            # Líneas internas de frentes (cuando hay más de uno)
            if frentes > 1 and w > 2.5:
                fw = w / frentes
                for f in range(1, frentes):
                    ax.plot([x + f * fw, x + f * fw], [0.10, 0.90],
                            color="white", linewidth=0.5, alpha=0.55, zorder=3)

            # Etiqueta de texto
            lines = [brand]
            if pd.notna(size) and size > 0:
                lines.append(f"{int(size)} ml")
            if pkg not in ("", "OTRO"):
                lines.append(pkg)
            label = "\n".join(lines)

            fs = max(4.0, min(7.5, w * 0.52))
            ax.text(x + w / 2 - 0.12, 0.50, label,
                    ha="center", va="center",
                    fontsize=fs, color="white", fontweight="bold",
                    linespacing=1.25, zorder=4, clip_on=True)

            x += w

    # ── Leyenda inferior ──────────────────────────────────────────────────
    handles = [
        mpatches.Patch(facecolor=c, label=p.replace("_", " ").title(), edgecolor="#888")
        for p, c in PROVIDER_COLORS.items()
    ]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=len(handles),
        bbox_to_anchor=(0.5, -0.025),
        fontsize=9,
        title="Proveedor",
        title_fontsize=10,
        framealpha=0.92,
        edgecolor="#BBBBBB"
    )

    plt.subplots_adjust(left=0.07, right=0.98, top=0.97, bottom=0.06)

    out_path = outdir / "planograma_v7_2.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close(fig)
    log(f"   Planograma PNG guardado → {out_path}")
    return out_path

# MAIN
def main():
    t0 = time.time()
    log("1) Cargando fuentes...")
    ejemplo, oxxo1, plano, caso = load_sources(".")

    log("2) Estandarizando fuentes...")
    hist = pd.concat([standardize_hist(ejemplo), standardize_hist(oxxo1)], ignore_index=True)
    hist = hist[(hist["PLANOGRUPO"] == TARGET_PLANOGRUPO) & (hist["DIRECCION_LEGO_ID"] == TARGET_DIRECTION)].copy()
    geom = standardize_geometry(plano, caso)
    log(f"   Historico filtrado: {len(hist):,} filas")
    log(f"   Formatos historicos: {hist['format_key'].nunique():,}")
    log(f"   Diseños planograma: {hist['design_ref'].nunique():,}")

    log("3) Construyendo instancia objetivo...")
    target_products, target_geom, target_hist = build_target_instance(hist, geom)
    log(f"   Target design_ref usado: {target_hist['design_ref'].mode().iloc[0]}")
    log(f"   Productos target: {len(target_products)}")
    log(f"   Charolas target: {len(target_geom)}")

    log("4) Construyendo tablas historicas...")
    tables = build_tables(hist, target_hist)

    model_bundle = None
    if USE_CHAROLA_MODEL:
        log("5) Entrenando predictor moderado de charola...")
        model_bundle = train_charola_model(tables["agg"])
    else:
        log("5) Predictor de charola desactivado.")

    log("6) Asignando productos a charolas...")
    scores_df = score_charolas(target_products, target_geom, tables, model_bundle)
    assigned = assign_charolas(target_products, scores_df)

    log("7) Ordenando dentro de charola con exacto por bloque + fallback heuristico...")
    pred_parts = []
    for ch, g in tqdm(assigned.groupby("pred_charola"), total=assigned["pred_charola"].nunique(), desc="Target charolas"):
        log(f"   Iniciando charola {int(ch)} con {len(g)} productos")
        c0 = time.time()
        pred_parts.append(solve_charola_hybrid(g.copy(), int(ch), tables))
        log(f"   Charola {int(ch)} terminada en {time.time()-c0:.1f}s")
    pred = pd.concat(pred_parts, ignore_index=True)

    metrics, detail, pb = evaluate_target(pred, target_products)
    outdir = Path(OUTPUT_DIR)
    outdir.mkdir(exist_ok=True, parents=True)

    metrics_full = {
        "config": {
            "TARGET_PLANOGRUPO": TARGET_PLANOGRUPO,
            "TARGET_DIRECTION": TARGET_DIRECTION,
            "TARGET_DESIGN_REF": TARGET_DESIGN_REF,
            "TARGET_SEGMENT": TARGET_SEGMENT,
            "USE_CHAROLA_MODEL": USE_CHAROLA_MODEL,
            "VERSION": "V7.2",
            "solver": "Exact-by-block + heuristic fallback for large charolas"
        },
        "holdout_metrics": None,
        "target_metrics": metrics,
        "weights": WEIGHTS
    }

    with open(outdir/"metrics_v7_2.json", "w", encoding="utf-8") as f:
        json.dump(metrics_full, f, ensure_ascii=False, indent=2)

    pred.to_csv(outdir/"target_recommendation_v7_2.csv", index=False)
    detail.to_csv(outdir/"target_evaluation_detail_v7_2.csv", index=False)
    pb.to_csv(outdir/"provider_block_metrics_v7_2.csv", index=False)
    assigned.to_csv(outdir/"charola_assignment_v7_2.csv", index=False)
    target_products.to_csv(outdir/"target_products_input_v7_2.csv", index=False)
    target_geom.to_csv(outdir/"target_geometry_v7_2.csv", index=False)

    log("8) Generando visualización del planograma...")   # ← NUEVO
    visualize_planogram(pred, target_geom, outdir)        # ← NUEVO

    log("9) Terminado.")
    log(f"   Archivos guardados en: {outdir.resolve()}")
    log(f"   Tiempo total: {time.time()-t0:.1f}s")
    print(json.dumps(metrics_full, ensure_ascii=False, indent=2))

if __name__ == "__main__":
    main()


[03:05:40] 1) Cargando fuentes...
[03:05:43] 2) Estandarizando fuentes...
[03:06:05]    Historico filtrado: 254,264 filas
[03:06:05]    Formatos historicos: 132
[03:06:06]    Diseños planograma: 132
[03:06:06] 3) Construyendo instancia objetivo...
[03:06:06]    Target design_ref usado: BCO_CF_Refrescos_3.5ID
[03:06:06]    Productos target: 334
[03:06:06]    Charolas target: 20
[03:06:06] 4) Construyendo tablas historicas...
[03:06:15] 5) Predictor de charola desactivado.
[03:06:15] 6) Asignando productos a charolas...
[03:06:15] 7) Ordenando dentro de charola con exacto por bloque + fallback heuristico...


Target charolas:   0%|          | 0/20 [00:00<?, ?it/s]

[03:06:15]    Iniciando charola 1 con 27 productos
[03:06:15]    Charola 1 terminada en 0.0s
[03:06:15]    Iniciando charola 2 con 10 productos
[03:06:15]    Charola 2 terminada en 0.0s
[03:06:15]    Iniciando charola 3 con 33 productos
[03:06:15]    Charola 3 terminada en 0.4s
[03:06:15]    Iniciando charola 4 con 26 productos
[03:06:16]    Charola 4 terminada en 0.2s
[03:06:16]    Iniciando charola 5 con 31 productos
[03:06:16]    Charola 5 terminada en 0.3s
[03:06:16]    Iniciando charola 6 con 26 productos
[03:06:16]    Charola 6 terminada en 0.3s
[03:06:16]    Iniciando charola 7 con 17 productos
[03:06:16]    Charola 7 terminada en 0.0s
[03:06:16]    Iniciando charola 8 con 34 productos
[03:06:16]    Charola 8 terminada en 0.2s
[03:06:16]    Iniciando charola 9 con 6 productos
[03:06:16]    Charola 9 terminada en 0.0s
[03:06:16]    Iniciando charola 10 con 11 productos
[03:06:16]    Charola 10 terminada en 0.0s
[03:06:16]    Iniciando charola 11 con 18 productos
[03:06:16]    Cha